# Figure S2

Draws supplementary drought maps and hydroclimatic drivers.


In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
from pyproj import Transformer
from scipy.ndimage import generic_filter
from shapely.geometry import LineString


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'outputs' / 'RECON_MAIN_2011_2023').exists():
            return candidate
    raise FileNotFoundError('Could not find outputs/RECON_MAIN_2011_2023 from the current working directory.')


def display_path(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


ROOT = find_repo_root()
RECON = ROOT / 'outputs' / 'RECON_MAIN_2011_2023'
OUT_DIR = ROOT / 'outputs' / 'figures' / 'FigS2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RECON_MATRIX_PATH = RECON / 'reconstruction' / 'wtd_reconstructed_matrix.npy'
MONTH_INDEX_PATH = RECON / 'metadata' / 'month_index.csv'
GRID_LOOKUP_PATH = RECON / 'metadata' / 'grid_lookup.csv'
MRVA_BOUNDARY_PATH = ROOT / 'assets' / 'spatial' / 'mrva_boundary.geojson'
MISSISSIPPI_RIVER_GMT_PATH = ROOT / 'assets' / 'spatial' / 'mississippi_river.gmt'

SWB_MONTHLY_PATH = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H6' / 'swb_monthly.csv'
PUMPING_MONTHLY_PATH = ROOT / 'data' / 'train_val_test_inputs' / 'GNN_spacetime' / 'H6' / 'aiwum_monthly.csv'
PRECIP_DIR = ROOT / 'data' / '10 precipitation' / 'daymet_prcp_monthly_v4r1_mrva'
PRECIP_MATRIX_PATH = PRECIP_DIR / 'daymet_prcp_monthly_mrva_1km.npy'
PRECIP_MONTH_INDEX_PATH = PRECIP_DIR / 'month_index.csv'

DROUGHT_YEARS = (2012, 2022, 2023)
DECLINE_MAP_YEARS = (2017, 2022, 2023)
PRE_DROUGHT_MONTHS_BY_YEAR = {
    2012: (1, 4),
    2017: (1, 5),
    2022: (1, 5),
    2023: (1, 5),
}
DROUGHT_MONTHS_BY_YEAR = {
    2012: (5, 10),
    2017: (6, 11),
    2022: (6, 11),
    2023: (6, 11),
}
MAP_VMAX_PERCENTILE = 98.5
MAP_DISPLAY_MEDIAN_FILTER_SIZE = 3
GRID_CRS = 'EPSG:5070'
LONLAT_CRS = 'EPSG:4326'
MAP_BOUNDARY_COLOR = '#1f1f1f'
MAP_BOUNDARY_LW = 0.45
MAP_RIVER_COLOR = '#A9DCEF'
MAP_RIVER_LW = 0.45
NETINF_BAR_COLOR = '#9BD9CC'
PUMPING_BAR_COLOR = '#8A8176'
PRECIP_BAR_COLOR = '#CFE8C9'
NETINF_LABEL_COLOR = '#3B8F83'
PUMPING_LABEL_COLOR = '#5F584F'
PRECIP_LABEL_COLOR = '#5D8F58'
BAR_EDGE_COLOR = '#1A1A1A'
MONTH_BAR_WIDTH_DAYS = 23
PANEL_GRID_COLOR = '#D6D6D6'
PANEL_SPINE_COLOR = '#2A2A2A'
DROUGHT_SHADE_COLOR = '#C97C7C'
DROUGHT_SHADE_ALPHA = 0.055
EXPORT_DPI = 600

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.dpi': EXPORT_DPI,
    'savefig.bbox': 'tight',
    'axes.linewidth': 0.65,
    'xtick.major.width': 0.55,
    'ytick.major.width': 0.55,
    'xtick.major.size': 2.5,
    'ytick.major.size': 2.5,
})

print('Reconstruction:', display_path(RECON))
print('Output:', display_path(OUT_DIR))


In [ ]:
mat = np.load(RECON_MATRIX_PATH, mmap_mode='r')
month_index = pd.read_csv(MONTH_INDEX_PATH)
month_index['date'] = pd.to_datetime(month_index['month_label'])
grid_lookup = pd.read_csv(GRID_LOOKUP_PATH)


def grid_edge_lonlat() -> tuple[np.ndarray, np.ndarray]:
    x_by_col = grid_lookup.groupby('col')['x'].first().sort_index().to_numpy(dtype=np.float64)
    y_by_row = grid_lookup.groupby('row')['y'].first().sort_index().to_numpy(dtype=np.float64)
    dx = float(np.nanmedian(np.diff(x_by_col)))
    dy = float(np.nanmedian(np.diff(y_by_row)))
    x_edges = np.concatenate([[x_by_col[0] - 0.5 * dx], x_by_col + 0.5 * dx])
    y_edges = np.concatenate([[y_by_row[0] - 0.5 * dy], y_by_row + 0.5 * dy])
    xx, yy = np.meshgrid(x_edges, y_edges)
    transformer = Transformer.from_crs(GRID_CRS, LONLAT_CRS, always_xy=True)
    lon, lat = transformer.transform(xx, yy)
    return np.asarray(lon), np.asarray(lat)


def plot_line_geometry_lonlat(ax, geometry, **kwargs) -> None:
    if geometry.is_empty:
        return
    geom_type = geometry.geom_type
    if geom_type == 'LineString':
        x, y = geometry.xy
        ax.plot(np.asarray(x), np.asarray(y), **kwargs)
    elif geom_type in {'MultiLineString', 'GeometryCollection'}:
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part, **kwargs)
    elif geom_type == 'Polygon':
        plot_line_geometry_lonlat(ax, geometry.boundary, **kwargs)
    elif geom_type == 'MultiPolygon':
        for part in geometry.geoms:
            plot_line_geometry_lonlat(ax, part.boundary, **kwargs)


def load_mrva_boundary_polygon():
    boundary = gpd.read_file(MRVA_BOUNDARY_PATH).to_crs(LONLAT_CRS)
    if hasattr(boundary.geometry, 'union_all'):
        return boundary.geometry.union_all()
    return boundary.unary_union


def read_gmt_segments(path: Path) -> list[np.ndarray]:
    segments = []
    current = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if current:
                segments.append(np.asarray(current, dtype=np.float64))
                current = []
            continue
        lon, lat = line.split()[:2]
        current.append((float(lon), float(lat)))
    if current:
        segments.append(np.asarray(current, dtype=np.float64))
    return segments


def load_mississippi_segments_lonlat() -> list[np.ndarray]:
    return read_gmt_segments(MISSISSIPPI_RIVER_GMT_PATH)


MAP_LON_EDGES, MAP_LAT_EDGES = grid_edge_lonlat()
MAP_EXTENT_LONLAT = (
    float(np.nanmin(MAP_LON_EDGES)),
    float(np.nanmax(MAP_LON_EDGES)),
    float(np.nanmin(MAP_LAT_EDGES)),
    float(np.nanmax(MAP_LAT_EDGES)),
)
MRVA_POLYGON_GEOMETRY = load_mrva_boundary_polygon()
MRVA_BOUNDARY_GEOMETRY = MRVA_POLYGON_GEOMETRY.boundary
MISSISSIPPI_SEGMENTS_LONLAT = load_mississippi_segments_lonlat()


def clipped_segment_to_mrva(segment: np.ndarray):
    if len(segment) < 2:
        return None
    return LineString(segment).intersection(MRVA_POLYGON_GEOMETRY)


def month_window_indices(year: int, month_start: int, month_end: int) -> np.ndarray:
    mask = (
        (month_index['date'].dt.year == year)
        & month_index['date'].dt.month.between(month_start, month_end)
    )
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        raise ValueError(f'No months found for {year}-{month_start:02d} to {year}-{month_end:02d}.')
    return idx


def compute_peak_decline(year: int) -> np.ndarray:
    pre_idx = month_window_indices(year, *PRE_DROUGHT_MONTHS_BY_YEAR[year])
    drought_idx = month_window_indices(year, *DROUGHT_MONTHS_BY_YEAR[year])
    pre_shallow = np.nanmin(np.asarray(mat[pre_idx, :], dtype=np.float32), axis=0)
    drought_deep = np.nanmax(np.asarray(mat[drought_idx, :], dtype=np.float32), axis=0)
    decline = (drought_deep - pre_shallow).astype(np.float32)
    return decline


def rasterize_grid_values(values: np.ndarray) -> np.ndarray:
    n_rows = int(grid_lookup['row'].max()) + 1
    n_cols = int(grid_lookup['col'].max()) + 1
    out = np.full((n_rows, n_cols), np.nan, dtype=np.float32)
    out[grid_lookup['row'].to_numpy(dtype=int), grid_lookup['col'].to_numpy(dtype=int)] = values
    return out


def nanmedian_quiet(window: np.ndarray) -> float:
    finite = window[np.isfinite(window)]
    if finite.size == 0:
        return np.nan
    return float(np.median(finite))


def smooth_display_raster(raster: np.ndarray, filter_size: int = MAP_DISPLAY_MEDIAN_FILTER_SIZE) -> np.ndarray:
    if filter_size <= 1:
        return raster
    display = generic_filter(raster, nanmedian_quiet, size=filter_size, mode='nearest')
    display[~np.isfinite(raster)] = np.nan
    return display.astype(np.float32)


def build_decline_rasters() -> tuple[dict[int, np.ndarray], float]:
    decline_rasters = {}
    all_values = []
    for year in DECLINE_MAP_YEARS:
        decline = compute_peak_decline(year)
        decline_rasters[year] = rasterize_grid_values(decline)
        all_values.append(decline[np.isfinite(decline)])

    common_values = np.concatenate(all_values)
    vmax = float(np.nanpercentile(common_values, MAP_VMAX_PERCENTILE))
    return decline_rasters, max(vmax, 1.0)


def plot_decline_map(raster: np.ndarray, year: int, vmax: float, *, show_colorbar: bool = False) -> Path:
    cmap = LinearSegmentedColormap.from_list(
        'mrva_loss',
        ['#F7F7F7', '#F2B07D', '#B64342'],
        N=256,
    )
    cmap.set_bad('#FFFFFF')
    display_raster = smooth_display_raster(raster)

    fig_width = 3.35 if show_colorbar else 3.0
    fig, ax = plt.subplots(figsize=(fig_width, 5.8), dpi=EXPORT_DPI)
    image = ax.pcolormesh(
        MAP_LON_EDGES,
        MAP_LAT_EDGES,
        display_raster,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        shading='flat',
        rasterized=True,
    )
    for segment in MISSISSIPPI_SEGMENTS_LONLAT:
        clipped = clipped_segment_to_mrva(segment)
        if clipped is not None:
            plot_line_geometry_lonlat(ax, clipped, color=MAP_RIVER_COLOR, lw=MAP_RIVER_LW, alpha=0.95, zorder=3)
    plot_line_geometry_lonlat(ax, MRVA_BOUNDARY_GEOMETRY, color=MAP_BOUNDARY_COLOR, lw=MAP_BOUNDARY_LW, zorder=4)
    ax.set_xlim(MAP_EXTENT_LONLAT[0], MAP_EXTENT_LONLAT[1])
    ax.set_ylim(MAP_EXTENT_LONLAT[2], MAP_EXTENT_LONLAT[3])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.tick_params(axis='both', which='both', bottom=False, left=False, labelbottom=False, labelleft=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    mean_lat = 0.5 * (MAP_EXTENT_LONLAT[2] + MAP_EXTENT_LONLAT[3])
    ax.set_aspect(1.0 / np.cos(np.deg2rad(mean_lat)))
    if show_colorbar:
        cbar = fig.colorbar(image, ax=ax, fraction=0.040, pad=0.025, extend='both')
        cbar.set_ticks([0, 1, 2, 3])
        cbar.set_ticklabels(['0', '1', '2', '3'])
        cbar.set_label('Maximum WTD decline (m)', rotation=90, labelpad=7)
        cbar.ax.tick_params(length=2.8, width=0.65, labelsize=10.5)
        cbar.outline.set_linewidth(0.6)
    out = OUT_DIR / f'FigS2_decline_{year}.png'
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out

print(f'Reconstruction matrix: {mat.shape[0]} months x {mat.shape[1]} grid cells')


In [ ]:
def compute_monthly_net_infiltration_mean() -> pd.DataFrame:
    swb = pd.read_csv(SWB_MONTHLY_PATH, usecols=['month_label', 'monthly_sum_net_infiltration'])
    out = (
        swb.groupby('month_label', as_index=False)['monthly_sum_net_infiltration']
        .mean()
        .sort_values('month_label')
    )
    out['date'] = pd.to_datetime(out['month_label'])
    return out.rename(columns={'monthly_sum_net_infiltration': 'net_infiltration_mean_mm'})[['date', 'net_infiltration_mean_mm']]


def compute_monthly_pumping_total() -> pd.DataFrame:
    pumping = pd.read_csv(PUMPING_MONTHLY_PATH, usecols=['month_label', 'monthly_pumping'])
    out = pumping.groupby('month_label', as_index=False)['monthly_pumping'].sum().sort_values('month_label')
    out['date'] = pd.to_datetime(out['month_label'])
    return out.rename(columns={'monthly_pumping': 'pumping_out_total_m3'})[['date', 'pumping_out_total_m3']]


def compute_monthly_precip_mean() -> pd.DataFrame:
    precip_month_index = pd.read_csv(PRECIP_MONTH_INDEX_PATH)
    precip_month_index['date'] = pd.to_datetime(precip_month_index['month_label'])
    precip = np.load(PRECIP_MATRIX_PATH, mmap_mode='r')
    precip_mean = np.nanmean(np.asarray(precip, dtype=np.float32), axis=1)
    out = precip_month_index[['date']].copy()
    out['precipitation_mean_mm'] = precip_mean
    return out


def build_driver_timeseries() -> pd.DataFrame:
    series = month_index[['date']].copy()
    series = series.merge(compute_monthly_net_infiltration_mean(), on='date', how='left')
    series = series.merge(compute_monthly_pumping_total(), on='date', how='left')
    series = series.merge(compute_monthly_precip_mean(), on='date', how='left')
    return series


def annotate_drought_years(ax: plt.Axes) -> None:
    for year in DROUGHT_YEARS:
        start_month, end_month = DROUGHT_MONTHS_BY_YEAR[year]
        start_date = pd.Timestamp(year, start_month, 1)
        end_date = pd.Timestamp(year, end_month, 1) + pd.offsets.MonthEnd(0)
        ax.axvspan(start_date, end_date, color=DROUGHT_SHADE_COLOR, alpha=DROUGHT_SHADE_ALPHA, lw=0)


def style_driver_axis(ax: plt.Axes, *, show_bottom_labels: bool = False) -> None:
    ax.set_axisbelow(True)
    ax.grid(axis='both', color=PANEL_GRID_COLOR, lw=0.45, alpha=0.62, zorder=0)
    for side in ['top', 'right', 'bottom', 'left']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color(PANEL_SPINE_COLOR)
        ax.spines[side].set_linewidth(0.72)
    ax.tick_params(
        axis='x',
        which='major',
        top=True,
        bottom=True,
        labeltop=False,
        labelbottom=show_bottom_labels,
        direction='out',
        length=3.0,
        width=0.65,
        pad=2.0,
    )
    ax.tick_params(
        axis='y',
        which='major',
        right=True,
        labelright=False,
        direction='out',
        length=3.0,
        width=0.65,
        pad=2.0,
    )


def plot_hydroclimatic_drivers(series: pd.DataFrame) -> Path:
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(7.0, 5.0),
        dpi=EXPORT_DPI,
        sharex=True,
        gridspec_kw={'hspace': 0.14},
    )
    fig.subplots_adjust(left=0.165, right=0.965, top=0.965, bottom=0.17)
    for ax in axes:
        style_driver_axis(ax, show_bottom_labels=(ax is axes[-1]))
        annotate_drought_years(ax)

    axes[0].bar(
        series['date'], series['precipitation_mean_mm'], width=MONTH_BAR_WIDTH_DAYS,
        color=PRECIP_BAR_COLOR, edgecolor='none', linewidth=0.0, align='center', zorder=3,
    )
    axes[0].axhline(0, color=BAR_EDGE_COLOR, lw=0.65)
    axes[0].set_ylabel('Precipitation\n(mm)')
    axes[0].yaxis.label.set_color(PRECIP_LABEL_COLOR)

    axes[1].bar(
        series['date'], series['net_infiltration_mean_mm'], width=MONTH_BAR_WIDTH_DAYS,
        color=NETINF_BAR_COLOR, edgecolor='none', linewidth=0.0, align='center', zorder=3,
    )
    axes[1].axhline(0, color='#666666', lw=0.45)
    axes[1].set_ylabel('Net infiltration\n(mm)')
    axes[1].yaxis.label.set_color(NETINF_LABEL_COLOR)

    axes[2].bar(
        series['date'], series['pumping_out_total_m3'], width=MONTH_BAR_WIDTH_DAYS,
        color=PUMPING_BAR_COLOR, edgecolor='none', linewidth=0.0, align='center', zorder=3,
    )
    axes[2].axhline(0, color='#666666', lw=0.45)
    axes[2].set_ylabel('Pumping\n(10$^9$ m$^3$)')
    axes[2].yaxis.label.set_color(PUMPING_LABEL_COLOR)
    axes[2].yaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value / 1e9:g}'))
    axes[2].xaxis.set_major_locator(mdates.YearLocator(1))
    axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[2].set_xlim(pd.Timestamp('2011-01-01'), pd.Timestamp('2023-12-31'))
    for tick in axes[2].get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha('right')

    out = OUT_DIR / 'FigS2_hydroclimatic_drivers.png'
    fig.savefig(out, dpi=EXPORT_DPI)
    plt.close(fig)
    return out


In [ ]:
decline_rasters, common_vmax = build_decline_rasters()
fig_paths = [
    plot_decline_map(decline_rasters[year], year, common_vmax, show_colorbar=True)
    for year in DECLINE_MAP_YEARS
]

driver_series = build_driver_timeseries()
fig_paths.append(plot_hydroclimatic_drivers(driver_series))

expected_names = [
    'FigS2_decline_2017.png',
    'FigS2_decline_2022.png',
    'FigS2_decline_2023.png',
    'FigS2_hydroclimatic_drivers.png',
]
if [path.name for path in fig_paths] != expected_names:
    raise AssertionError('Unexpected Fig. S2_1 output list.')

print('FigS2 outputs written to:')
for path in fig_paths:
    print(' ', display_path(path))


In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

for path in fig_paths:
    print(display_path(path))
    if Image is not None:
        display(Image(filename=str(path)))
